In [32]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import torch
import torchaudio
from torchaudio import transforms
import os
from pathlib import Path
import gc
import re
from difflib import SequenceMatcher

# Define paths
download_path = Path('data/DATA/output_chunks')
metadata_file = download_path / 'metadata.csv'
excel_file = Path('data/FINAL DATA.xlsx')
new_leq_file = Path('new_leq_results.csv')
locations_file = Path('locations_linked.csv')

# Load metadata
try:
    df = pd.read_csv(metadata_file)
    print("Columns in metadata.csv:", df.columns.tolist())
    print("Sample metadata rows:", df.head().to_dict(orient='records'))
except FileNotFoundError:
    raise FileNotFoundError(f"metadata.csv not found at {metadata_file}")

# Ensure required columns
required_columns = ['road', 'classID', 'class', 'slice_file_name']
if not all(col in df.columns for col in required_columns):
    missing = [col for col in required_columns if col not in df.columns]
    raise KeyError(f"Missing required columns in metadata.csv: {missing}")

# Create relative_path
if 'relative_path' not in df.columns:
    df['relative_path'] = df['road'].astype(str) + '/' + df['slice_file_name'].astype(str)
df['relative_path'] = df['relative_path'].apply(lambda x: str(Path(x)))
df = df[['relative_path', 'classID', 'road', 'class', 'slice_file_name']].copy()
df['road'] = df['road'].str.upper().str.strip()
print("Columns in df after processing:", df.columns.tolist())
print("Sample relative_path values:", df['relative_path'].head().tolist())
print("Unique locations from metadata:", df['road'].unique().tolist())

# Load locations_linked.csv with speed data
try:
    locations_df = pd.read_csv(locations_file)
    locations_df.columns = locations_df.columns.str.strip()
    
    # Define expected columns
    expected_columns = ['location', 'speed', 'location_2']
    if not all(col in locations_df.columns for col in expected_columns):
        raise ValueError(f"locations_linked.csv must contain columns: {expected_columns}. Found: {locations_df.columns.tolist()}")
    
    locations_df['location'] = locations_df['location'].str.upper().str.strip()
    locations_df['location_2'] = locations_df['location_2'].str.upper().str.strip()
    locations_df['speed'] = pd.to_numeric(locations_df['speed'], errors='coerce')
    
    # Create dictionaries for matching
    location_speed_dict = dict(zip(locations_df['location_2'], locations_df['speed']))  # Primary matching on location_2
    location_speed_dict_loc1 = dict(zip(locations_df['location'], locations_df['speed']))  # Secondary matching on location
    location_to_location2 = dict(zip(locations_df['location'], locations_df['location_2']))
    
    def simplify_name(name):
        if pd.isna(name):
            return ""
        name_str = str(name).upper()
        # Remove coordinates (handle spaces in coordinates)
        name_str = re.sub(r'\(\s*\d+°\s*\d+[_ ]\d+\s*[NSEW]\s*\d+°\s*\d+[_ ]\d+\s*[NSEW]\s*\)', '', name_str)
        # Remove other parentheses
        name_str = re.sub(r'\([^)]*\)', '', name_str)
        # Remove common prefixes and suffixes
        prefixes = ['AROUND', 'OPP\\.', 'NEAR', 'NEXT TO', 'CLOSE TO']
        suffixes = ['ROAD', 'RD', 'STREET', 'ST', 'AVENUE', 'AVE']  # Exclude HOSPITAL, MALL, SCHOOL to preserve key terms
        for prefix in prefixes:
            name_str = re.sub(rf'^{prefix}\s+', '', name_str)
        for suffix in suffixes:
            name_str = re.sub(rf'\s+{suffix}$', '', name_str)
        # Remove special characters
        name_str = re.sub(r'[^\w\s]', '', name_str)
        # Normalize spaces
        name_str = re.sub(r'\s+', ' ', name_str).strip()
        return name_str
    
    def extract_base_name(name):
        simplified = simplify_name(name)
        base = re.sub(r'\s+\d+$', '', simplified).strip()  # Remove trailing numbers
        return base
    
    # Simplified and base name dictionaries
    location_speed_dict_simple = {}
    location_base_groups = {}
    for _, row in locations_df.iterrows():
        loc, speed, loc2 = row['location'], row['speed'], row['location_2']
        # Simplify both location and location_2
        simple_key_loc = simplify_name(loc)
        simple_key_loc2 = simplify_name(loc2)
        base_name_loc = extract_base_name(loc)
        base_name_loc2 = extract_base_name(loc2)
        
        # Store in simple lookup
        location_speed_dict_simple.setdefault(simple_key_loc, []).append((loc, speed))
        location_speed_dict_simple.setdefault(simple_key_loc2, []).append((loc2, speed))
        
        # Store in base groups
        location_base_groups.setdefault(base_name_loc, []).append((loc, speed, simple_key_loc))
        location_base_groups.setdefault(base_name_loc2, []).append((loc2, speed, simple_key_loc2))
    
    print(f"Loaded locations_linked.csv with columns: {locations_df.columns.tolist()}")
    print(f"Total locations in locations_linked.csv: {len(location_speed_dict)}")
    print("\nAvailable locations in locations_linked.csv (location):")
    for loc, speed in sorted(location_speed_dict_loc1.items()):
        print(f"  - {loc}: {speed:.2f} km/h")
    print("\nAvailable locations in locations_linked.csv (location_2):")
    for loc, speed in sorted(location_speed_dict.items()):
        print(f"  - {loc}: {speed:.2f} km/h")
    
    # Check matching
    unique_metadata_locations = df['road'].unique()
    print(f"\nUnique locations in metadata.csv: {len(unique_metadata_locations)}")
    matched = []
    unmatched = []
    for loc in unique_metadata_locations:
        simple_loc = simplify_name(loc)
        if loc in location_speed_dict or loc in location_speed_dict_loc1 or simple_loc in location_speed_dict_simple:
            matched.append(f"{loc} (matched to {location_speed_dict_simple.get(simple_loc, [(loc, None)])[0][0] if simple_loc in location_speed_dict_simple else loc})")
        else:
            unmatched.append(loc)
    
    print(f"\nWill match: {len(matched)} locations")
    for loc in matched:
        print(f"  ✓ {loc}")
    if unmatched:
        print(f"\nWill NOT match: {len(unmatched)} locations (will use average speed)")
        for loc in unmatched:
            print(f"  ✗ {loc}")
    
except FileNotFoundError:
    raise FileNotFoundError(f"locations_linked.csv not found at {locations_file}")
except Exception as e:
    raise Exception(f"Error loading locations_linked.csv: {e}")

# Load new Leq data
try:
    new_leq_df = pd.read_csv(new_leq_file)
    new_leq_df['Filename'] = new_leq_df['Filename'].str.replace('.wav', '', case=False).str.upper().str.strip()
    print("Columns in new_leq_results.csv:", new_leq_df.columns.tolist())
except FileNotFoundError:
    raise FileNotFoundError(f"new_leq_results.csv not found at {new_leq_file}")

# Load Excel data
try:
    df_leq = pd.read_excel(excel_file, sheet_name='Noise Leq Data')
    df_pcu = pd.read_excel(excel_file, sheet_name='PCU')
    df_pcu_conv = pd.read_excel(excel_file, sheet_name='PCU Conversion')

    # Process Noise Leq Data
    leq_hourly = df_leq.iloc[1:-2].reset_index(drop=True)
    header_row = leq_hourly.iloc[0]
    leq_hourly = leq_hourly.drop(0).reset_index(drop=True)
    leq_hourly.columns = header_row
    leq_hourly.iloc[:, 0] = leq_hourly.iloc[:, 0].str.upper().str.strip()
    
    # Parse time columns
    time_cols = leq_hourly.columns[1:]
    parsed_times = []
    for col in time_cols:
        original = str(col)
        substituted = re.sub(r'(\d+)-(\d+)(AM|PM)', r'\1:00\3', original)
        try:
            parsed_time = pd.to_datetime(substituted, format='%I:%M%p').time()
            parsed_times.append(parsed_time)
        except ValueError:
            parsed_times.append(original)
    leq_hourly.columns = [leq_hourly.columns[0]] + parsed_times
    leq_hourly.iloc[:, 1:] = leq_hourly.iloc[:, 1:].astype(float)
    
    # Process PCU data
    df_pcu = df_pcu.iloc[1:].reset_index(drop=True)
    header_row_pcu = df_pcu.iloc[0]
    df_pcu = df_pcu.drop(0).reset_index(drop=True)
    df_pcu.columns = header_row_pcu
    df_pcu.iloc[:, 0] = df_pcu.iloc[:, 0].str.upper().str.strip()
    mask = df_pcu.iloc[:, 1:].apply(lambda x: x.astype(str).str.contains('AVERAGE', case=True, na=False) | 
                                    pd.to_numeric(x, errors='coerce').isna()).any(axis=1)
    df_pcu = df_pcu[~mask]
    df_pcu.iloc[:, 1:] = df_pcu.iloc[:, 1:].astype(float)
    
    # Process PCU Conversion
    df_pcu_conv = df_pcu_conv.iloc[1:].reset_index(drop=True)
    header_row_pcu_conv = df_pcu_conv.iloc[0]
    df_pcu_conv = df_pcu_conv.drop(0).reset_index(drop=True)
    df_pcu_conv.columns = header_row_pcu_conv
    df_pcu_conv.iloc[:, 0] = df_pcu_conv.iloc[:, 0].str.upper().str.strip()
    mask_conv = df_pcu_conv.iloc[:, 1:].apply(lambda x: x.astype(str).str.contains('Vehicle count', case=True, na=False) | 
                                              pd.to_numeric(x, errors='coerce').isna()).any(axis=1)
    df_pcu_conv = df_pcu_conv[~mask_conv]
    df_pcu_conv.iloc[:, 1:] = df_pcu_conv.iloc[:, 1:].astype(float)
    
except FileNotFoundError:
    raise FileNotFoundError(f"{excel_file} not found.")
except Exception as e:
    raise Exception(f"Error loading Excel data: {e}")

# AudioUtil class
class AudioUtil:
    @staticmethod
    def open(audio_file):
        sig, sr = torchaudio.load(audio_file)
        return (sig, sr)

    @staticmethod
    def resample(aud, newsr=22050):
        sig, sr = aud
        if sr == newsr:
            return aud
        resig = torchaudio.transforms.Resample(sr, newsr)(sig)
        return (resig, newsr)

    @staticmethod
    def pad_trunc(aud, max_ms=4000):
        sig, sr = aud
        num_rows, sig_len = sig.shape
        max_len = sr // 1000 * max_ms
        if sig_len > max_len:
            sig = sig[:, :max_len]
        elif sig_len < max_len:
            pad_len = max_len - sig_len
            pad = torch.zeros((num_rows, pad_len))
            sig = torch.cat((sig, pad), 1)
        return (sig, sr)

    @staticmethod
    def mfcc_feature(aud, n_mfcc=40, n_fft=1024, hop_len=512):
        sig, sr = aud
        mfcc = torchaudio.transforms.MFCC(
            sample_rate=sr,
            n_mfcc=n_mfcc,
            melkwargs={"n_fft": n_fft, "hop_length": hop_len, "n_mels": 64}
        )(sig)
        mfcc_db = torchaudio.transforms.AmplitudeToDB(top_db=80)(mfcc)
        return mfcc_db.mean(dim=2).squeeze(0)

# Helper functions
def extract_time_slot(filename):
    match = re.search(r'(\d{1,2}-\d{1,2}[ap]m|\d{1,2}-\d{1,2})', filename.lower())
    return match.group(1) if match else None

def adjust_for_doppler(leq, speed, distance=10.0):
    c = 343.0
    if speed is None or np.isnan(speed):
        return leq
    relative_velocity = speed / 3.6
    freq_ratio = c / (c - relative_velocity) if relative_velocity > 0 else c / (c + abs(relative_velocity))
    delta_leq = 10 * np.log10(freq_ratio)
    adjusted_leq = leq + delta_leq
    return max(adjusted_leq, leq - 5.0)

# Process audio files
data = []
hours = ['6-7AM', '7-8AM', '8-9AM', '9-10AM', '10-11AM', '11-12PM', '12-1PM', '1-2PM', '2-3PM', '3-4PM', '4-5PM', '5-6PM']
speed_match_stats = {'matched': 0, 'fallback_avg': 0}
location_occurrence_tracker = {}

# Debug mode for specific locations
debug_locations = [
    'AROUND LANGATA HOSPITAL (1°19_40_S 36°47_21_E)',
    'IMAARA MALL (1° 19_ 41_S 36°52_49_E)',
    'JEVANJEE MOI AVENUE (1° 16_ 51_ S 36°49_ 13_ E)',
    'KAREN C SCHOOL (1°20_19_S 36°44_27_E)',
    'LANGATA LINK 1°19_19_S 36°47_01_E',
    'LIKONI RD (1°17_55_S 36°50_14_E)',
    'NGONG ROAD (1° 18_ 22_S 36° 44_ 24_E)',
    'NYAYO LA RD (1°18_20_ S 36°49_21_E)',
    'OPP. KU HOSPITAL (1°10_34_ S 36°54_47_E)'
]

for idx, row in df.iterrows():
    relative_path = row['relative_path']
    audio_file = download_path / relative_path
    if not audio_file.exists():
        print(f"Audio file not found: {audio_file}, skipping.")
        continue
    
    try:
        # Process audio
        aud = AudioUtil.open(audio_file)
        aud = AudioUtil.resample(aud)
        aud = AudioUtil.pad_trunc(aud)
        mfcc = AudioUtil.mfcc_feature(aud)
        mfcc_np = mfcc.numpy()
        mean_mfcc = np.mean(mfcc_np)
        std_mfcc = np.std(mfcc_np)
        
        # Match Leq data
        road = row['road']
        time_slot = extract_time_slot(row['slice_file_name'])
        leq_value = None
        if road in new_leq_df['Filename'].values:
            leq_value = new_leq_df.loc[new_leq_df['Filename'] == road, 'Leq_dBA'].iloc[0]
        elif road in leq_hourly.iloc[:, 0].values:
            row_idx = leq_hourly.index[leq_hourly.iloc[:, 0] == road].tolist()[0]
            if time_slot and time_slot in hours:
                time_idx = hours.index(time_slot)
                leq_value = leq_hourly.iloc[row_idx, time_idx + 1]
            else:
                leq_value = leq_hourly.iloc[row_idx, 1:].mean()
        
        # Match speed data
        speed_value = None
        match_method = None
        debug = road in debug_locations
        if debug:
            print(f"\nDEBUG: Matching road '{road}'")
        
        if road in location_speed_dict:
            speed_value = location_speed_dict[road]
            speed_match_stats['matched'] += 1
            match_method = "exact (location_2)"
            if debug:
                print(f"  - Exact match (location_2): '{road}' -> {speed_value:.2f} km/h")
        elif road in location_speed_dict_loc1:
            speed_value = location_speed_dict_loc1[road]
            speed_match_stats['matched'] += 1
            match_method = "exact (location)"
            if debug:
                print(f"  - Exact match (location): '{road}' -> {speed_value:.2f} km/h")
        else:
            simple_road = simplify_name(road)
            base_road = extract_base_name(road)
            if debug:
                print(f"  - Simplified: '{simple_road}', Base: '{base_road}'")
            
            if simple_road in location_speed_dict_simple:
                matched_location, speed_value = location_speed_dict_simple[simple_road][0]
                speed_match_stats['matched'] += 1
                match_method = f"fuzzy (matched to '{matched_location}')"
                if debug:
                    print(f"  - Fuzzy match: '{simple_road}' -> '{matched_location}' ({speed_value:.2f} km/h)")
            elif base_road in location_base_groups:
                group = location_base_groups[base_road]
                occurrence_idx = location_occurrence_tracker.get(base_road, 0)
                matched_location, speed_value, _ = group[min(occurrence_idx, len(group) - 1)]
                speed_match_stats['matched'] += 1
                match_method = f"semantic-ordered #{occurrence_idx + 1} (matched to '{matched_location}')"
                location_occurrence_tracker[base_road] = occurrence_idx + 1
                if debug:
                    print(f"  - Semantic-ordered match: '{base_road}' -> '{matched_location}' ({speed_value:.2f} km/h)")
            else:
                # Fuzzy matching with similarity score
                best_score = 0
                best_match = None
                for loc in list(location_speed_dict_loc1.keys()) + list(location_speed_dict.keys()):
                    score = SequenceMatcher(None, simple_road, simplify_name(loc)).ratio()
                    if debug:
                        print(f"  - Comparing '{simple_road}' to '{simplify_name(loc)}': score = {score:.2f}")
                    if score > best_score and score > 0.55:  # Lowered threshold
                        best_score = score
                        best_match = loc
                if best_match:
                    speed_value = location_speed_dict_loc1.get(best_match) or location_speed_dict.get(best_match)
                    speed_match_stats['matched'] += 1
                    match_method = f"fuzzy-similarity (matched to '{best_match}', score: {best_score:.2f})"
                    if debug:
                        print(f"  - Fuzzy-similarity match: '{simple_road}' -> '{best_match}' ({speed_value:.2f} km/h)")
                else:
                    speed_value = np.nanmean(list(location_speed_dict.values()))
                    speed_match_stats['fallback_avg'] += 1
                    match_method = "average fallback"
                    if speed_match_stats['fallback_avg'] <= 5 or debug:
                        print(f"⚠ Location '{road}' not found, using average speed: {speed_value:.2f} km/h")
        
        if speed_match_stats['matched'] <= 10 or debug:
            print(f"✓ Matched '{road}' via {match_method}: {speed_value:.2f} km/h")
        
        # Apply Doppler effect
        adjusted_leq = adjust_for_doppler(leq_value, speed_value) if leq_value is not None and speed_value is not None else leq_value
        
        # Store data
        data.append({
            'relative_path': relative_path,
            'road': road,
            'classID': row['classID'],
            'class': row['class'],
            'Mean_MFCC': mean_mfcc,
            'Std_MFCC': std_mfcc,
            'Leq_dBA': leq_value,
            'Adjusted_Leq_dBA': adjusted_leq,
            'Speed_kmh': speed_value
        })
        
    except Exception as e:
        print(f"Error processing {audio_file}: {e}")
    finally:
        del aud, mfcc, mfcc_np
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

# Save results
mfcc_df = pd.DataFrame(data)
mfcc_df.to_csv('mfcc_stats_with_doppler.csv', index=False)
print("\n=== Speed Matching Statistics ===")
print(f"Successfully matched: {speed_match_stats['matched']}")
print(f"Used fallback average: {speed_match_stats['fallback_avg']}")
print(f"Total processed: {len(data)}")
print("MFCC statistics saved to mfcc_stats_with_doppler.csv")

# Visualizations
plt.figure(figsize=(12, 6))
for class_val in mfcc_df['class'].unique():
    class_data = mfcc_df[mfcc_df['class'] == class_val]
    plt.bar(class_data['road'] + f' ({class_val})', class_data['Mean_MFCC'], yerr=class_data['Std_MFCC'], capsize=5, label=class_val, alpha=0.7)
plt.xlabel('Location (Vehicle Class)')
plt.ylabel('Mean MFCC Value (dB)')
plt.title('Average MFCC by Location and Vehicle Class')
plt.xticks(rotation=45, ha='right')
plt.legend()
plt.tight_layout()
plt.savefig('mfcc_by_location_class.png')
plt.close()

if not mfcc_df['Adjusted_Leq_dBA'].isna().all():
    plt.figure(figsize=(10, 6))
    scatter = plt.scatter(mfcc_df['Adjusted_Leq_dBA'], mfcc_df['Mean_MFCC'], c=mfcc_df['classID'], cmap='viridis')
    plt.colorbar(scatter, label='Class ID')
    plt.xlabel('Adjusted Leq (dBA)')
    plt.ylabel('Mean MFCC Value (dB)')
    plt.title('Mean MFCC vs Adjusted Leq by Vehicle Class')
    plt.grid(True)
    plt.savefig('mfcc_vs_adjusted_leq_class.png')
    plt.close()
else:
    print("No Adjusted Leq data available for visualization.")

Columns in metadata.csv: ['slice_file_name', 'fsID', 'start', 'end', 'road', 'classID', 'class']
Sample metadata rows: [{'slice_file_name': 'Around Arya School(1°16_32_ S 36°49_29_ E)--1-0.wav', 'fsID': 'Around Arya School(1°16_32_ S 36°49_29_ E)', 'start': 0.0, 'end': 6.0, 'road': 'Around Arya School(1°16_32_ S 36°49_29_ E)', 'classID': 7, 'class': 'heavy truck'}, {'slice_file_name': 'Around Arya School(1°16_32_ S 36°49_29_ E)--1-1.wav', 'fsID': 'Around Arya School(1°16_32_ S 36°49_29_ E)', 'start': 6.0, 'end': 12.0, 'road': 'Around Arya School(1°16_32_ S 36°49_29_ E)', 'classID': 1, 'class': 'motorcycle'}, {'slice_file_name': 'Around Arya School(1°16_32_ S 36°49_29_ E)--1-2.wav', 'fsID': 'Around Arya School(1°16_32_ S 36°49_29_ E)', 'start': 12.0, 'end': 18.0, 'road': 'Around Arya School(1°16_32_ S 36°49_29_ E)', 'classID': 1, 'class': 'motorcycle'}, {'slice_file_name': 'Around Arya School(1°16_32_ S 36°49_29_ E)--1-3.wav', 'fsID': 'Around Arya School(1°16_32_ S 36°49_29_ E)', 'start

c:\Users\USER\Music\SonusAI\venv\Lib\site-packages\torchaudio\_backend\utils.py:213: UserWarning: In 2.9, this function's implementation will be changed to use torchaudio.load_with_torchcodec` under the hood. Some parameters like ``normalize``, ``format``, ``buffer_size``, and ``backend`` will be ignored. We recommend that you port your code to rely directly on TorchCodec's decoder instead: https://docs.pytorch.org/torchcodec/stable/generated/torchcodec.decoders.AudioDecoder.html#torchcodec.decoders.AudioDecoder.
  warnings.warn(


✓ Matched 'AROUND ARYA SCHOOL(1°16_32_ S 36°49_29_ E)' via fuzzy (matched to 'AROUND ARYA SCHOOL'): 37.71 km/h
✓ Matched 'AROUND ARYA SCHOOL(1°16_32_ S 36°49_29_ E)' via fuzzy (matched to 'AROUND ARYA SCHOOL'): 37.71 km/h
✓ Matched 'AROUND ARYA SCHOOL(1°16_32_ S 36°49_29_ E)' via fuzzy (matched to 'AROUND ARYA SCHOOL'): 37.71 km/h
✓ Matched 'AROUND ARYA SCHOOL(1°16_32_ S 36°49_29_ E)' via fuzzy (matched to 'AROUND ARYA SCHOOL'): 37.71 km/h
✓ Matched 'AROUND ARYA SCHOOL(1°16_32_ S 36°49_29_ E)' via fuzzy (matched to 'AROUND ARYA SCHOOL'): 37.71 km/h
✓ Matched 'AROUND ARYA SCHOOL(1°16_32_ S 36°49_29_ E)' via fuzzy (matched to 'AROUND ARYA SCHOOL'): 37.71 km/h
✓ Matched 'AROUND ARYA SCHOOL(1°16_32_ S 36°49_29_ E)' via fuzzy (matched to 'AROUND ARYA SCHOOL'): 37.71 km/h
✓ Matched 'AROUND ARYA SCHOOL(1°16_32_ S 36°49_29_ E)' via fuzzy (matched to 'AROUND ARYA SCHOOL'): 37.71 km/h
✓ Matched 'AROUND ARYA SCHOOL(1°16_32_ S 36°49_29_ E)' via fuzzy (matched to 'AROUND ARYA SCHOOL'): 37.71 km/h
✓

In [34]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
from pathlib import Path

# Try to set a compatible Matplotlib style, fallback to default if unavailable
try:
    plt.style.use('seaborn-v0_8')  # Updated style for newer Matplotlib versions
except OSError:
    plt.style.use('default')  # Fallback to Matplotlib's default style
    print("Warning: 'seaborn-v0_8' style not found, using default Matplotlib style.")

# Load the dataset
data_path = Path('mfcc_stats_with_doppler.csv')
try:
    mfcc_df = pd.read_csv(data_path)
except FileNotFoundError:
    raise FileNotFoundError(f"Dataset not found at {data_path}")

# Ensure required columns are present
required_columns = ['road', 'class', 'Mean_MFCC', 'Adjusted_Leq_dBA', 'Speed_kmh']
if not all(col in mfcc_df.columns for col in required_columns):
    missing = [col for col in required_columns if col not in mfcc_df.columns]
    raise KeyError(f"Missing columns: {missing}")

# Figure 1: Box Plot of Mean_MFCC by Vehicle Class
plt.figure(figsize=(12, 6))
sns.boxplot(x='class', y='Mean_MFCC', data=mfcc_df, palette='Set2')
plt.title('Distribution of Mean MFCC by Vehicle Class', fontsize=14)
plt.xlabel('Vehicle Class', fontsize=12)
plt.ylabel('Mean MFCC (dB)', fontsize=12)
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.savefig('boxplot_mfcc_by_class.png')
plt.close()
print("Saved: boxplot_mfcc_by_class.png")

# Figure 2: Scatter Plot of Adjusted_Leq_dBA vs. Mean_MFCC
plt.figure(figsize=(10, 6))
scatter = sns.scatterplot(x='Adjusted_Leq_dBA', y='Mean_MFCC', hue='class', size='Speed_kmh', 
                         sizes=(20, 200), data=mfcc_df, palette='viridis', alpha=0.7)
plt.title('Adjusted Leq vs. Mean MFCC by Vehicle Class', fontsize=14)
plt.xlabel('Adjusted Leq (dBA)', fontsize=12)
plt.ylabel('Mean MFCC (dB)', fontsize=12)
plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left', title='Vehicle Class')
plt.grid(True)
plt.tight_layout()
plt.savefig('scatter_leq_vs_mfcc.png')
plt.close()
print("Saved: scatter_leq_vs_mfcc.png")

# Figure 3: Bar Plot of Average Adjusted_Leq_dBA by Location
# Aggregate mean Adjusted_Leq_dBA by road, sort by value, and select top 10
location_leq = mfcc_df.groupby('road')['Adjusted_Leq_dBA'].mean().sort_values(ascending=False).head(10).reset_index()
plt.figure(figsize=(12, 6))
sns.barplot(x='Adjusted_Leq_dBA', y='road', data=location_leq, palette='Blues_d')
plt.title('Top 10 Locations by Average Adjusted Leq', fontsize=14)
plt.xlabel('Adjusted Leq (dBA)', fontsize=12)
plt.ylabel('Location', fontsize=12)
plt.tight_layout()
plt.savefig('bar_leq_by_location.png')
plt.close()
print("Saved: bar_leq_by_location.png")

# Figure 4: Correlation Heatmap
# Compute correlation matrix for numerical columns
correlation_matrix = mfcc_df[['Mean_MFCC', 'Std_MFCC', 'Adjusted_Leq_dBA', 'Speed_kmh']].corr()
plt.figure(figsize=(8, 6))
sns.heatmap(correlation_matrix, annot=True, cmap='coolwarm', vmin=-1, vmax=1, center=0, 
            annot_kws={'size': 10}, cbar_kws={'label': 'Correlation Coefficient'})
plt.title('Correlation Matrix of MFCC, Leq, and Speed', fontsize=14)
plt.tight_layout()
plt.savefig('heatmap_correlation.png')
plt.close()
print("Saved: heatmap_correlation.png")

# Figure 5: Histogram of Speed_kmh by Vehicle Class
plt.figure(figsize=(12, 6))
for vehicle_class in mfcc_df['class'].unique():
    sns.histplot(mfcc_df[mfcc_df['class'] == vehicle_class]['Speed_kmh'], 
                 label=vehicle_class, bins=20, alpha=0.5, stat='count')
plt.title('Speed Distribution by Vehicle Class', fontsize=14)
plt.xlabel('Speed (km/h)', fontsize=12)
plt.ylabel('Count', fontsize=12)
plt.legend(title='Vehicle Class')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('histogram_speed_by_class.png')
plt.close()
print("Saved: histogram_speed_by_class.png")

print("\nAll visualizations generated successfully.")

C:\Users\USER\AppData\Local\Temp\ipykernel_51564\1516723867.py:29: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.

  sns.boxplot(x='class', y='Mean_MFCC', data=mfcc_df, palette='Set2')


Saved: boxplot_mfcc_by_class.png
Saved: scatter_leq_vs_mfcc.png


C:\Users\USER\AppData\Local\Temp\ipykernel_51564\1516723867.py:57: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `y` variable to `hue` and set `legend=False` for the same effect.

  sns.barplot(x='Adjusted_Leq_dBA', y='road', data=location_leq, palette='Blues_d')


Saved: bar_leq_by_location.png
Saved: heatmap_correlation.png
Saved: histogram_speed_by_class.png

All visualizations generated successfully.
